# PaddleOCR – Musterlösung

## Setup

In [ ]:
import os
import time
import numpy as np
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["FLAGS_allocator_strategy"] = "auto_growth"
os.environ["PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK"] = "True"

from paddleocr import PaddleOCR
print("✅ Imports erfolgreich!")


## Modell laden

In [ ]:
MODES = {
    "Multilingual (DE/EN/FR/CH/...)": {
        "lang": "german",
        "rec_mobile": "PP-OCRv5_mobile_rec",
        "rec_server": "PP-OCRv5_server_rec",
    },
    "Arabic": {
        "lang": "ar",
        "rec_mobile": "arabic_PP-OCRv5_mobile_rec",
        "rec_server": "arabic_PP-OCRv5_mobile_rec",
    },
}

def get_model(mode="Multilingual (DE/EN/FR/CH/...)", mobile=True):
    cfg = MODES[mode]
    rec_model = cfg["rec_mobile"] if mobile else cfg["rec_server"]
    kwargs = dict(
        lang=cfg["lang"],
        device="cpu",
        use_doc_orientation_classify=False,
        use_doc_unwarping=False,
        use_textline_orientation=False,
        text_recognition_model_name=rec_model,
    )
    if mobile:
        kwargs["text_detection_model_name"] = "PP-OCRv5_mobile_det"
    return PaddleOCR(**kwargs)

model = get_model()
print("✅ Modell geladen!")


## Lösung 1: Bild laden

In [ ]:
image = np.array(Image.open("test_imgs/general_ocr.png").convert("RGB"))

plt.figure(figsize=(10, 8))
plt.imshow(image)
plt.axis("off")
plt.title(f"Bild geladen: {image.shape[1]}x{image.shape[0]} Pixel")
plt.show()


## OCR ausführen

In [ ]:
start = time.perf_counter()
results = list(model.predict(image.copy()))
elapsed = time.perf_counter() - start

print(f"✅ Fertig in {elapsed*1000:.0f}ms")
print(f"   Anzahl Ergebnisse: {len(results)}")


## Lösung 2: Detections bauen

In [ ]:
detections = []

for result in results:
    for poly, text, score in zip(
        result.get("rec_polys", []),
        result.get("rec_texts", []),
        result.get("rec_scores", []),
    ):
        detections.append({
            "text": text,
            "confidence": round(float(score), 4),
            "polygon": [[int(p[0]), int(p[1])] for p in poly],
        })

print(f"Gefundene Texte: {len(detections)}")
if detections:
    print("Erstes Element:", detections[0])


## Lösung 3: Farbe berechnen

In [ ]:
def conf_to_color(conf):
    r = int((1 - conf) * 255)
    g = int(conf * 255)
    return (r, g, 0)

print("conf=1.0 →", conf_to_color(1.0), " (sollte (0, 255, 0) sein)")
print("conf=0.0 →", conf_to_color(0.0), " (sollte (255, 0, 0) sein)")
print("conf=0.5 →", conf_to_color(0.5), " (sollte (127, 127, 0) sein)")


## Lösung 4: Boxen zeichnen

In [ ]:
def build_image(image_rgb, detections):
    img = Image.fromarray(image_rgb)
    draw = ImageDraw.Draw(img)

    for det in detections:
        poly = [tuple(p) for p in det["polygon"]]
        conf = det.get("confidence", 0)
        color = conf_to_color(conf)
        draw.polygon(poly, outline=color, width=2)

    return np.array(img)

result_img = build_image(image, detections)
plt.figure(figsize=(12, 10))
plt.imshow(result_img)
plt.axis("off")
plt.title(f"{len(detections)} Texte erkannt in {elapsed*1000:.0f}ms")
plt.show()


## Bonus: Texte ausgeben

In [ ]:
for det in detections:
    print(f'"{det["text"]}" ({det["confidence"]:.0%})')
